In [1]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda")

def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", " ", code)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

train = pd.read_json("../data/train.jsonl", lines=True)
valid = pd.read_json("../data/valid.jsonl", lines=True)
for d in (train, valid):
    d["clean"] = d["func"].apply(clean_code)

vectorizer = TfidfVectorizer(
    token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|->|&&|\|\||[^\s\w]",
    lowercase=False, ngram_range=(1, 2), min_df=2, max_features=50000,
)
X_train = vectorizer.fit_transform(train["clean"])
X_valid = vectorizer.transform(valid["clean"])
y_train = train["target"].values
y_valid = valid["target"].values
print(X_train.shape)

(21854, 50000)


In [2]:
def to_batch(X, idx):
    return torch.tensor(X[idx].toarray(), dtype=torch.float32, device=device)

def predict(model, X, batch=512):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch):
            idx = np.arange(i, min(i + batch, X.shape[0]))
            out.append(model(to_batch(X, idx)).argmax(1).cpu().numpy())
    return np.concatenate(out)

model = nn.Sequential(
    nn.Linear(50000, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2)
).to(device)

counts = np.bincount(y_train)
weights = torch.tensor(len(y_train) / (2 * counts), dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

best_f1, best_state = 0, None
for epoch in range(1, 11):
    model.train()
    perm = np.random.permutation(X_train.shape[0])
    for i in range(0, len(perm), 256):
        idx = perm[i:i + 256]
        xb = to_batch(X_train, idx)
        yb = torch.tensor(y_train[idx], device=device)
        opt.zero_grad()
        loss_fn(model(xb), yb).backward()
        opt.step()
    p = predict(model, X_valid)
    acc, f1 = accuracy_score(y_valid, p), f1_score(y_valid, p)
    print(f"epoch {epoch}: val acc={acc:.3f}, F1(vuln)={f1:.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

epoch 1: val acc=0.625, F1(vuln)=0.607
epoch 2: val acc=0.636, F1(vuln)=0.615
epoch 3: val acc=0.626, F1(vuln)=0.573
epoch 4: val acc=0.625, F1(vuln)=0.593
epoch 5: val acc=0.623, F1(vuln)=0.616
epoch 6: val acc=0.617, F1(vuln)=0.571
epoch 7: val acc=0.615, F1(vuln)=0.610
epoch 8: val acc=0.610, F1(vuln)=0.584
epoch 9: val acc=0.605, F1(vuln)=0.589
epoch 10: val acc=0.607, F1(vuln)=0.578


In [3]:
from sklearn.metrics import confusion_matrix

test = pd.read_json("../data/test.jsonl", lines=True)
test["clean"] = test["func"].apply(clean_code)
X_test = vectorizer.transform(test["clean"])
y_test = test["target"].values

model.load_state_dict(best_state)
p = predict(model, X_test)
print(classification_report(y_test, p, digits=3))
print(confusion_matrix(y_test, p))

              precision    recall  f1-score   support

           0      0.651     0.574     0.610      1486
           1      0.555     0.632     0.591      1246

    accuracy                          0.601      2732
   macro avg      0.603     0.603     0.600      2732
weighted avg      0.607     0.601     0.601      2732

[[853 633]
 [458 788]]
